### Check Code Data Preprocessing

In [1]:
import pandas as pd

In [2]:
df1 = pd.read_csv("/root/MIMICIV/src/mimiciv_clinical_dataset_tabular_death_visit.csv")
df2 = pd.read_csv("/root/MIMICIV/src/mimiciv_clinical_dataset_tabular_death_visit_2.csv")

In [3]:
df1.equals(df2)

True

Then /root/MIMICIV/src/mimiciv_clinical_dataset_tabular_death_visit.csv has been created using the tabular.py script. Let's go further and see how the dataset landmark_df.csv has been created.

In [6]:
# Importing libraries (exactly as from xgboost.ipynb)
import ast
import xgboost as xgb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.sparse as sp

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from scipy.sparse import hstack

In [7]:
# Load dataset
df = pd.read_csv("mimiciv_clinical_dataset_tabular_death_visit.csv")

# Count admission categories
admission_counts = df['admission_category'].value_counts()

# Display counts
print("Admission category counts:")
print(admission_counts := df['admission_category'].value_counts())

Admission category counts:
admission_category
NORMAL       291667
EMERGENCY    254361
Name: count, dtype: int64


In [8]:
def safe_join(x):
    if pd.isna(x) or x in ['[]', '', 'None', 'NaN', 'na', 'nan']:
        return ''
    try:
        x_eval = ast.literal_eval(x)
        if isinstance(x_eval, list):
            return '; '.join(x_eval)
        else:
            return ''
    except:
        return ''

landmark_rows = []

#df = df.sort_values(['subject_id', 'hadm_id']).reset_index(drop=True) # That's not the right ordering!
# the dataset is still ordered, however we should consider a more robust way of doing so.

In [9]:
for patient_id, group in df.groupby('subject_id'):
    group = group.reset_index(drop=True)
    n_visits = group.shape[0]

    for landmark_idx in range(n_visits):
        current_visit = group.iloc[landmark_idx]
        past_visits = group.iloc[:landmark_idx + 1]

        # Here: to be more efficient - how to say confirmed/new? For now okay here

        unique_vals = set(past_visits['medication_list'])
        if len(unique_vals) == 1 and pd.isna(next(iter(unique_vals))):
            meds_text = ''
        else:
            meds_text = ' '.join(past_visits['medication_list'].apply(safe_join))
        
        unique_vals = set(past_visits['diagnosis_list'])
        if len(unique_vals) == 1 and pd.isna(next(iter(unique_vals))):
            diag_text = ''
        else:
            diag_text = ' '.join(past_visits['diagnosis_list'].apply(safe_join))
        
        unique_vals = set(past_visits['procedure_list'])
        if len(unique_vals) == 1 and pd.isna(next(iter(unique_vals))):
            proc_text = ''
        else:
            proc_text = ' '.join(past_visits['procedure_list'].apply(safe_join))

        # Define outcome
        days_to_death = current_visit['days_from_last_visit_to_death']
        death_in_90days = 1 if (not np.isnan(days_to_death) and days_to_death <= 90) else 0

        landmark_rows.append({
            'subject_id': current_visit['subject_id'],
            'landmark_visit': landmark_idx + 1,
            'age_at_landmark': current_visit['age_at_event'],
            'gender': current_visit['gender'],
            'num_total_visits': landmark_idx + 1,
            'days_from_last_visit': current_visit['days_until_next_visit'],
            'death_in_90days': death_in_90days,
            'med_text': meds_text,
            'diag_text': diag_text,
            'proc_text': proc_text
        })

landmark_df = pd.DataFrame(landmark_rows)

In [10]:
landmark_df.head(10)

,subject_id,landmark_visit,age_at_landmark,gender,num_total_visits,days_from_last_visit,death_in_90days,med_text,diag_text,proc_text
0,10000032,1,52,F,1,50.0,1,Furosemide; Ipratropium Bromide Neb; Potassium...,Portal hypertension; Other ascites; Cirrhosis ...,Percutaneous abdominal drainage
1,10000032,2,52,F,2,25.0,1,Furosemide; Ipratropium Bromide Neb; Potassium...,Portal hypertension; Other ascites; Cirrhosis ...,Percutaneous abdominal drainage Percutaneous a...
2,10000032,3,52,F,3,11.0,1,Furosemide; Ipratropium Bromide Neb; Potassium...,Portal hypertension; Other ascites; Cirrhosis ...,Percutaneous abdominal drainage Percutaneous a...
3,10000032,4,52,F,4,-1.0,1,Furosemide; Ipratropium Bromide Neb; Potassium...,Portal hypertension; Other ascites; Cirrhosis ...,Percutaneous abdominal drainage Percutaneous a...
4,10000068,1,19,F,1,-1.0,0,,"Alcohol abuse, unspecified",Other nonoperative respiratory measurements
5,10000084,1,72,M,1,32.0,1,Pramipexole; Pravastatin; rivastigmine; Senna;...,Neurocognitive disorder with Lewy bodies; Deme...,
6,10000084,2,72,M,2,-1.0,1,Pramipexole; Pravastatin; rivastigmine; Senna;...,Neurocognitive disorder with Lewy bodies; Deme...,
7,10000108,1,25,M,1,-1.0,0,,Cellulitis and abscess of oral soft tissues; O...,
8,10000117,1,55,F,1,672.0,0,Heparin; Sodium Chloride 0.9% Flush,"Dysphagia, unspecified; Other specified sympto...",
9,10000117,2,57,F,2,-1.0,0,Heparin; Sodium Chloride 0.9% Flush Iso-Osmot...,"Dysphagia, unspecified; Other specified sympto...",Reposition Left Upper Femur with Internal Fix...


In [31]:
# Convert gender explicitly into binary numeric values
landmark_df['gender_numeric'] = landmark_df['gender'].map({'F': 0, 'M': 1})
landmark_df.head(5)

,subject_id,landmark_visit,age_at_landmark,gender,num_total_visits,days_from_last_visit,death_in_90days,med_text,diag_text,proc_text,gender_numeric
0,10000032,1,52,F,1,50.0,1,Furosemide; Ipratropium Bromide Neb; Potassium...,Portal hypertension; Other ascites; Cirrhosis ...,Percutaneous abdominal drainage,0
1,10000032,2,52,F,2,25.0,1,Furosemide; Ipratropium Bromide Neb; Potassium...,Portal hypertension; Other ascites; Cirrhosis ...,Percutaneous abdominal drainage Percutaneous a...,0
2,10000032,3,52,F,3,11.0,1,Furosemide; Ipratropium Bromide Neb; Potassium...,Portal hypertension; Other ascites; Cirrhosis ...,Percutaneous abdominal drainage Percutaneous a...,0
3,10000032,4,52,F,4,-1.0,1,Furosemide; Ipratropium Bromide Neb; Potassium...,Portal hypertension; Other ascites; Cirrhosis ...,Percutaneous abdominal drainage Percutaneous a...,0
4,10000068,1,19,F,1,-1.0,0,,"Alcohol abuse, unspecified",Other nonoperative respiratory measurements,0


In [32]:
landmark_df.to_csv('landmark_df_2.csv', index=False, na_rep = 'Unknown')

In [18]:
# Let us check if they are equals
landmark_df_1 = pd.read_csv("landmark_df.csv")

In [33]:
landmark_df.equals(landmark_df_1)

False

In [34]:
landmark_df_1.iloc[44]

subject_id                                                            10001186
landmark_visit                                                               1
age_at_landmark                                                             46
gender                                                                       F
num_total_visits                                                             1
death_in_90days                                                              0
med_text                     Sodium Chloride 0.9%  Flush Ondansetron Guaife...
diag_text                    Disruption of external operation (surgical) wo...
proc_text                                               Removal of skull plate
gender_numeric                                                               0
days_since_previous_visit                                                 -1.0
Name: 44, dtype: object

In [35]:
landmark_df.iloc[44]

subject_id                                                       10001186
landmark_visit                                                          1
age_at_landmark                                                        46
gender                                                                  F
num_total_visits                                                        1
days_from_last_visit                                                 23.0
death_in_90days                                                         0
med_text                Sodium Chloride 0.9%  Flush; Ondansetron; Guai...
diag_text               Disruption of external operation (surgical) wo...
proc_text                                          Removal of skull plate
gender_numeric                                                          0
Name: 44, dtype: object

Still not equals. Something is missing...

In [36]:
# Ensure your DataFrame is sorted by patient and landmark_visit
landmark_df = landmark_df.sort_values(['subject_id', 'landmark_visit']).reset_index(drop=True)

# Function to perform transformation
def transform_to_past_variable(group):
    group = group.sort_values('landmark_visit').copy()
    # Shift the values down (future to past)
    group['days_since_last_visit'] = group['days_from_last_visit'].shift(1)
    # First landmark always has -1
    group['days_since_last_visit'].iloc[0] = -1
    return group

# Apply the transformation per patient
landmark_df = landmark_df.groupby('subject_id').apply(transform_to_past_variable).reset_index(drop=True)

# Drop the original future-looking column (optional but strongly recommended)
landmark_df.drop(columns=['days_from_last_visit'], inplace=True)

# Check the result clearly
print(landmark_df[['subject_id', 'landmark_visit', 'days_since_last_visit']].head(15))

    subject_id  landmark_visit  days_since_last_visit
0     10000032               1                   -1.0
1     10000032               2                   50.0
2     10000032               3                   25.0
3     10000032               4                   11.0
4     10000068               1                   -1.0
5     10000084               1                   -1.0
6     10000084               2                   32.0
7     10000108               1                   -1.0
8     10000117               1                   -1.0
9     10000117               2                  672.0
10    10000161               1                   -1.0
11    10000248               1                   -1.0
12    10000280               1                   -1.0
13    10000560               1                   -1.0
14    10000635               1                   -1.0


In [37]:
landmark_df = landmark_df.rename(columns={'days_since_last_visit':'days_since_previous_visit'})
landmark_df.head(10)

,subject_id,landmark_visit,age_at_landmark,gender,num_total_visits,death_in_90days,med_text,diag_text,proc_text,gender_numeric,days_since_previous_visit
0,10000032,1,52,F,1,1,Furosemide; Ipratropium Bromide Neb; Potassium...,Portal hypertension; Other ascites; Cirrhosis ...,Percutaneous abdominal drainage,0,-1.0
1,10000032,2,52,F,2,1,Furosemide; Ipratropium Bromide Neb; Potassium...,Portal hypertension; Other ascites; Cirrhosis ...,Percutaneous abdominal drainage Percutaneous a...,0,50.0
2,10000032,3,52,F,3,1,Furosemide; Ipratropium Bromide Neb; Potassium...,Portal hypertension; Other ascites; Cirrhosis ...,Percutaneous abdominal drainage Percutaneous a...,0,25.0
3,10000032,4,52,F,4,1,Furosemide; Ipratropium Bromide Neb; Potassium...,Portal hypertension; Other ascites; Cirrhosis ...,Percutaneous abdominal drainage Percutaneous a...,0,11.0
4,10000068,1,19,F,1,0,,"Alcohol abuse, unspecified",Other nonoperative respiratory measurements,0,-1.0
5,10000084,1,72,M,1,1,Pramipexole; Pravastatin; rivastigmine; Senna;...,Neurocognitive disorder with Lewy bodies; Deme...,,1,-1.0
6,10000084,2,72,M,2,1,Pramipexole; Pravastatin; rivastigmine; Senna;...,Neurocognitive disorder with Lewy bodies; Deme...,,1,32.0
7,10000108,1,25,M,1,0,,Cellulitis and abscess of oral soft tissues; O...,,1,-1.0
8,10000117,1,55,F,1,0,Heparin; Sodium Chloride 0.9% Flush,"Dysphagia, unspecified; Other specified sympto...",,0,-1.0
9,10000117,2,57,F,2,0,Heparin; Sodium Chloride 0.9% Flush Iso-Osmot...,"Dysphagia, unspecified; Other specified sympto...",Reposition Left Upper Femur with Internal Fix...,0,672.0


In [38]:
landmark_df.to_csv('landmark_df_2.csv', index=False, na_rep = 'Unknown')

In [39]:
landmark_df_2 = pd.read_csv('landmark_df_2.csv')

In [40]:
landmark_df_1.head(10)

,subject_id,landmark_visit,age_at_landmark,gender,num_total_visits,death_in_90days,med_text,diag_text,proc_text,gender_numeric,days_since_previous_visit
0,10000032,1,52,F,1,1,Furosemide Ipratropium Bromide Neb Potassium C...,Portal hypertension Other ascites Cirrhosis of...,Percutaneous abdominal drainage,0,-1.0
1,10000032,2,52,F,2,1,Furosemide Ipratropium Bromide Neb Potassium C...,Portal hypertension Other ascites Cirrhosis of...,Percutaneous abdominal drainage Percutaneous a...,0,50.0
2,10000032,3,52,F,3,1,Furosemide Ipratropium Bromide Neb Potassium C...,Portal hypertension Other ascites Cirrhosis of...,Percutaneous abdominal drainage Percutaneous a...,0,25.0
3,10000032,4,52,F,4,1,Furosemide Ipratropium Bromide Neb Potassium C...,Portal hypertension Other ascites Cirrhosis of...,Percutaneous abdominal drainage Percutaneous a...,0,11.0
4,10000068,1,19,F,1,0,Unknown,"Alcohol abuse, unspecified",Other nonoperative respiratory measurements,0,-1.0
5,10000084,1,72,M,1,1,Pramipexole Pravastatin rivastigmine Senna QUE...,Neurocognitive disorder with Lewy bodies Demen...,Unknown,1,-1.0
6,10000084,2,72,M,2,1,Pramipexole Pravastatin rivastigmine Senna QUE...,Neurocognitive disorder with Lewy bodies Demen...,,1,32.0
7,10000108,1,25,M,1,0,Unknown,Cellulitis and abscess of oral soft tissues Ot...,Unknown,1,-1.0
8,10000117,1,55,F,1,0,Heparin Sodium Chloride 0.9% Flush,"Dysphagia, unspecified Other specified symptom...",Unknown,0,-1.0
9,10000117,2,57,F,2,0,Heparin Sodium Chloride 0.9% Flush Iso-Osmoti...,"Dysphagia, unspecified Other specified symptom...",Reposition Left Upper Femur with Internal Fix...,0,672.0


In [41]:
landmark_df_2.head(10)

,subject_id,landmark_visit,age_at_landmark,gender,num_total_visits,death_in_90days,med_text,diag_text,proc_text,gender_numeric,days_since_previous_visit
0,10000032,1,52,F,1,1,Furosemide; Ipratropium Bromide Neb; Potassium...,Portal hypertension; Other ascites; Cirrhosis ...,Percutaneous abdominal drainage,0,-1.0
1,10000032,2,52,F,2,1,Furosemide; Ipratropium Bromide Neb; Potassium...,Portal hypertension; Other ascites; Cirrhosis ...,Percutaneous abdominal drainage Percutaneous a...,0,50.0
2,10000032,3,52,F,3,1,Furosemide; Ipratropium Bromide Neb; Potassium...,Portal hypertension; Other ascites; Cirrhosis ...,Percutaneous abdominal drainage Percutaneous a...,0,25.0
3,10000032,4,52,F,4,1,Furosemide; Ipratropium Bromide Neb; Potassium...,Portal hypertension; Other ascites; Cirrhosis ...,Percutaneous abdominal drainage Percutaneous a...,0,11.0
4,10000068,1,19,F,1,0,NaN,"Alcohol abuse, unspecified",Other nonoperative respiratory measurements,0,-1.0
5,10000084,1,72,M,1,1,Pramipexole; Pravastatin; rivastigmine; Senna;...,Neurocognitive disorder with Lewy bodies; Deme...,NaN,1,-1.0
6,10000084,2,72,M,2,1,Pramipexole; Pravastatin; rivastigmine; Senna;...,Neurocognitive disorder with Lewy bodies; Deme...,NaN,1,32.0
7,10000108,1,25,M,1,0,NaN,Cellulitis and abscess of oral soft tissues; O...,NaN,1,-1.0
8,10000117,1,55,F,1,0,Heparin; Sodium Chloride 0.9% Flush,"Dysphagia, unspecified; Other specified sympto...",NaN,0,-1.0
9,10000117,2,57,F,2,0,Heparin; Sodium Chloride 0.9% Flush Iso-Osmot...,"Dysphagia, unspecified; Other specified sympto...",Reposition Left Upper Femur with Internal Fix...,0,672.0


In [42]:
landmark_df_2.dtypes
landmark_df_2.proc_text[5]

nan

In [43]:
landmark_df_1.dtypes
landmark_df_1.proc_text[5]

'Unknown'

In [44]:
landmark_df_2.to_csv('landmark_df_2.csv', index=False, na_rep = 'Unknown')

In [45]:
landmark_df_3 = pd.read_csv('landmark_df_2.csv')

In [46]:
landmark_df_3.head(20)

,subject_id,landmark_visit,age_at_landmark,gender,num_total_visits,death_in_90days,med_text,diag_text,proc_text,gender_numeric,days_since_previous_visit
0,10000032,1,52,F,1,1,Furosemide; Ipratropium Bromide Neb; Potassium...,Portal hypertension; Other ascites; Cirrhosis ...,Percutaneous abdominal drainage,0,-1.0
1,10000032,2,52,F,2,1,Furosemide; Ipratropium Bromide Neb; Potassium...,Portal hypertension; Other ascites; Cirrhosis ...,Percutaneous abdominal drainage Percutaneous a...,0,50.0
2,10000032,3,52,F,3,1,Furosemide; Ipratropium Bromide Neb; Potassium...,Portal hypertension; Other ascites; Cirrhosis ...,Percutaneous abdominal drainage Percutaneous a...,0,25.0
3,10000032,4,52,F,4,1,Furosemide; Ipratropium Bromide Neb; Potassium...,Portal hypertension; Other ascites; Cirrhosis ...,Percutaneous abdominal drainage Percutaneous a...,0,11.0
4,10000068,1,19,F,1,0,Unknown,"Alcohol abuse, unspecified",Other nonoperative respiratory measurements,0,-1.0
5,10000084,1,72,M,1,1,Pramipexole; Pravastatin; rivastigmine; Senna;...,Neurocognitive disorder with Lewy bodies; Deme...,Unknown,1,-1.0
6,10000084,2,72,M,2,1,Pramipexole; Pravastatin; rivastigmine; Senna;...,Neurocognitive disorder with Lewy bodies; Deme...,Unknown,1,32.0
7,10000108,1,25,M,1,0,Unknown,Cellulitis and abscess of oral soft tissues; O...,Unknown,1,-1.0
8,10000117,1,55,F,1,0,Heparin; Sodium Chloride 0.9% Flush,"Dysphagia, unspecified; Other specified sympto...",Unknown,0,-1.0
9,10000117,2,57,F,2,0,Heparin; Sodium Chloride 0.9% Flush Iso-Osmot...,"Dysphagia, unspecified; Other specified sympto...",Reposition Left Upper Femur with Internal Fix...,0,672.0


In [47]:
landmark_df_3.equals(landmark_df_1)

False

Okay, so the code here it's the one that has been generated trough xgboost.ipynb. The Unknown have been creating saving two times the dataframe. First, creating NaN and then creating Unknown.

#### Why not all blank spaces are 'Unknown'?
Let's take the 10000084 subject. In the second landmark visit we don't have anymore Unknown, but, rather, ''. We need to investigate further.
Solved! See the code above, has already the solution.